In [1]:
# ============================================================
# NACC EXTERNAL VALIDATION — Notebook setup
# ============================================================
# WHY: This is a fresh notebook for the NACC validation phase.
# We load the ADNI-trained model and feature lists we saved
# earlier — NACC work builds on those without needing the ADNI
# notebook's memory.

import pandas as pd
import numpy as np
import joblib
import os

# --- Paths ---
# Folder with your saved ADNI models
save_folder = r"C:\Users\Chima\OneDrive\Documents\Python Scripts\saved_models"
# Folder with your NACC CSV files (adjust to where NACC lives)
nacc_folder = r"C:\Users\Chima\Downloads"  # adjust to your NACC location

# --- Load the trained ADNI model and feature lists ---
minimal_model = joblib.load(os.path.join(save_folder, 'minimal_model_LR.joblib'))
minimal_features = joblib.load(os.path.join(save_folder, 'minimal_features.joblib'))

print("Loaded ADNI-trained minimal model.")
print(f"\nThe model expects these {len(minimal_features)} features:")
for i, f in enumerate(minimal_features, 1):
    print(f"  {i:2}. {f}")

Loaded ADNI-trained minimal model.

The model expects these 26 features:
   1. TOTSCORE
   2. TOTAL13
   3. MMSCORE
   4. CDRSB
   5. FAQTOTAL
   6. GDTOTAL
   7. RAVLT_learning_total
   8. AVDEL30MIN
   9. AVDELTOT
  10. TRAASCOR
  11. TRABSCOR
  12. CATANIMSC
  13. LIMMTOTAL
  14. LDELTOTAL
  15. PTEDUCAT
  16. AGE
  17. APOE4_count
  18. Hippocampus_total
  19. Entorhinal_total
  20. Ventricle_total
  21. Fusiform_total
  22. ICV
  23. sex_female
  24. race_nonwhite
  25. ethnicity_hispanic
  26. married


In [2]:
# ============================================================
# NACC STEP 1 — Load diagnosis & identifier columns
# ============================================================
# WHY: We start exactly as we did with ADNI — by understanding
# the diagnosis structure so we can build the MCI cohort and the
# conversion label. We load ONLY the columns needed for this,
# keeping memory low despite the large file.

import pandas as pd
import numpy as np
import os

# Adjust this to where your main NACC file lives
nacc_path = r"C:\Users\Chima\Downloads\investigator_nacc73.csv"

# Columns needed for cohort + label construction
cohort_cols = [
    'NACCID',        # patient ID
    'NACCVNUM',      # visit number (1 = first visit)
    'VISITYR', 'VISITMO',  # visit year and month (for timing)
    'NACCUDSD',      # diagnosis: 1=Normal 2=Impaired 3=MCI 4=Dementia
    'NACCALZD',      # is Alzheimer's the cause of impairment
]

# Load only those columns from the big file
# First confirm they all exist
preview = pd.read_csv(nacc_path, nrows=5)
available = [c for c in cohort_cols if c in preview.columns]
missing = [c for c in cohort_cols if c not in preview.columns]
print(f"Found: {available}")
if missing:
    print(f"MISSING (check names): {missing}")

dx = pd.read_csv(nacc_path, usecols=available, low_memory=False)
print(f"\nLoaded: {len(dx):,} rows, {dx['NACCID'].nunique():,} unique patients")

# Inspect the diagnosis distribution
print(f"\nNACCUDSD distribution (all visits):")
print(dx['NACCUDSD'].value_counts(dropna=False).sort_index())
print("  1=Normal  2=Impaired-not-MCI  3=MCI  4=Dementia")

# Inspect visit numbering
print(f"\nVisits per patient (first 10 visit numbers):")
print(dx['NACCVNUM'].value_counts(dropna=False).sort_index().head(10))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Chima\\Downloads\\investigator_nacc73.csv'

In [3]:
# ============================================================
# Find the NACC file on your system
# ============================================================
# WHY: The path I guessed was wrong. This searches your common
# folders for the NACC CSV files so we can point to the right one.

import os

# Search these common locations
search_roots = [
    r"C:\Users\Chima\Downloads",
    r"C:\Users\Chima\OneDrive\Pictures\Desktop",
    r"C:\Users\Chima\OneDrive\Desktop",
    r"C:\Users\Chima\Desktop",
    r"C:\Users\Chima\Documents",
    r"C:\Users\Chima\OneDrive\Documents",
]

print("Searching for NACC files...\n")
for root in search_roots:
    if os.path.exists(root):
        for dirpath, dirnames, filenames in os.walk(root):
            nacc_files = [f for f in filenames
                          if 'nacc' in f.lower() and f.endswith('.csv')]
            if nacc_files:
                print(f"Found in: {dirpath}")
                for f in nacc_files:
                    size_mb = os.path.getsize(os.path.join(dirpath, f)) / (1024*1024)
                    print(f"   {f}  ({size_mb:.0f} MB)")
                print()

Searching for NACC files...

Found in: C:\Users\Chima\OneDrive\Pictures\Desktop\Alzheimer's Project
   investigator_fcsf_nacc73.csv  (0 MB)
   investigator_mri_nacc73.csv  (17 MB)

Found in: C:\Users\Chima\OneDrive\Documents\Python Scripts
   investigator_nacc73.csv  (951 MB)



In [4]:
# ============================================================
# NACC STEP 1 — Load diagnosis & identifier columns
# ============================================================
# WHY: We start as we did with ADNI — understanding the diagnosis
# structure to build the MCI cohort and conversion label. We load
# only the needed columns to keep memory low on this large file.

import pandas as pd
import numpy as np
import os

# Correct path to the main NACC file
nacc_path = r"C:\Users\Chima\OneDrive\Documents\Python Scripts\investigator_nacc73.csv"

# Columns needed for cohort + label construction
cohort_cols = [
    'NACCID',        # patient ID
    'NACCVNUM',      # visit number (1 = first visit)
    'VISITYR', 'VISITMO',  # visit year and month (for timing)
    'NACCUDSD',      # diagnosis: 1=Normal 2=Impaired 3=MCI 4=Dementia
    'NACCALZD',      # is Alzheimer's the cause of impairment
]

# Confirm columns exist before loading the full file
preview = pd.read_csv(nacc_path, nrows=5)
available = [c for c in cohort_cols if c in preview.columns]
missing = [c for c in cohort_cols if c not in preview.columns]
print(f"Found: {available}")
if missing:
    print(f"MISSING (check names): {missing}")

dx = pd.read_csv(nacc_path, usecols=available, low_memory=False)
print(f"\nLoaded: {len(dx):,} rows, {dx['NACCID'].nunique():,} unique patients")

# Diagnosis distribution
print(f"\nNACCUDSD distribution (all visits):")
print(dx['NACCUDSD'].value_counts(dropna=False).sort_index())
print("  1=Normal  2=Impaired-not-MCI  3=MCI  4=Dementia")

# Visit numbering
print(f"\nVisits per patient (first 10 visit numbers):")
print(dx['NACCVNUM'].value_counts(dropna=False).sort_index().head(10))

Found: ['NACCID', 'NACCVNUM', 'NACCUDSD', 'NACCALZD']
MISSING (check names): ['VISITYR', 'VISITMO']

Loaded: 214,976 rows, 56,532 unique patients

NACCUDSD distribution (all visits):
NACCUDSD
1    106475
2      9575
3     37957
4     60945
8        24
Name: count, dtype: int64
  1=Normal  2=Impaired-not-MCI  3=MCI  4=Dementia

Visits per patient (first 10 visit numbers):
NACCVNUM
1     56532
2     39365
3     29006
4     21676
5     16526
6     12877
7      9852
8      7384
9      5722
10     4446
Name: count, dtype: int64


In [5]:
# ============================================================
# Find NACC's visit date/timing columns
# ============================================================
# WHY: VISITYR/VISITMO weren't found. NACC names its visit timing
# columns something else. We need them to measure the 24-month
# conversion window. Let's search the full column list for
# date/time-related columns.

# Load just the header to see all columns
preview = pd.read_csv(nacc_path, nrows=5)

# Hunt for anything date or visit-timing related
print("Columns containing 'VIS', 'YR', 'MO', 'DATE', or 'DAY':")
candidates = [c for c in preview.columns
              if any(k in c.upper() for k in
                     ['VIS', 'YR', 'MO', 'DATE', 'DAY'])]
for c in candidates:
    print(f"  {c}")

Columns containing 'VIS', 'YR', 'MO', 'DATE', or 'DAY':
  VISITDATE
  NACCDAYS
  NACCMOD
  NACCDSMO
  NACCDSYR
  NACCNRMO
  NACCNRYR
  FRMDATEA1
  MODEA1
  RMMODEA1
  BIRTHMO
  BIRTHYR
  ETHSYRIA
  ETHSAMOAN
  ETHCHAMOR
  FRMDATEA1A
  MODEA1A
  RMMODEA1A
  INCOMEYR
  EATLESSYR
  LESSMEDSYR
  FRMDATEA2
  MODEA2
  RMMODEA2
  INCNTMOD
  INVISITS
  INBIRMO
  INBIRYR
  FRMDATEA3
  MODEA3
  RMMODEA3
  NACCMOM
  FRMDATEA4
  MODEA4
  RMMODEA4
  FRMDATEA4A
  MODEA4A
  RMMODEA4A
  STARTMO1
  ENDMO1
  STARTMO2
  ENDMO2
  STARTMO3
  ENDMO3
  STARTMO4
  ENDMO4
  STARTMO5
  ENDMO5
  STARTMO6
  ENDMO6
  STARTMO7
  ENDMO7
  STARTMO8
  ENDMO8
  FRMDATEA5D2
  MODEA5D2
  RMMODEA5D2
  SMOKYRS
  QUITSMOK
  ALCFREQYR
  NACCSTYR
  NACCTIYR
  PDYR
  PDOTHRYR
  THYROID
  CANCCHEMO
  PULMONARY
  DEP2YRS
  BCPILLSYR
  FRMDATEB1
  MODEB1
  VISION
  VISCORR
  VISWCORR
  EMOT
  FRMDATEB3
  MODEB3
  HANDMOVR
  HANDMOVL
  FRMDATEB4
  MODEB4
  RMMODEB4
  MEMORY
  FRMDATEB5
  MODEB5
  RMMODEB5
  MOT
  MOTSEV
  FRMDATEB

In [6]:
# ============================================================
# NACC STEP 2 — Reload with the timing column and verify it
# ============================================================
# WHY: NACCDAYS = days from each patient's baseline visit. This
# is a NACC-derived convenience variable that does our date math
# for us. We reload including it and confirm it behaves as
# expected (0 at first visit, increasing at later visits).

cohort_cols = [
    'NACCID', 'NACCVNUM', 'NACCDAYS',
    'NACCUDSD', 'NACCALZD'
]

dx = pd.read_csv(nacc_path, usecols=cohort_cols, low_memory=False)
print(f"Loaded: {len(dx):,} rows, {dx['NACCID'].nunique():,} patients")

# Verify NACCDAYS at visit 1 should be ~0 for everyone
print(f"\nNACCDAYS at visit 1 (NACCVNUM==1) — should be ~0:")
print(dx[dx['NACCVNUM'] == 1]['NACCDAYS'].describe())

# And NACCDAYS should grow at later visits
print(f"\nNACCDAYS at visit 2 — should be larger (~365+):")
print(dx[dx['NACCVNUM'] == 2]['NACCDAYS'].describe())

# Look at one patient's full trajectory as a sanity check
example_id = dx[dx['NACCVNUM'] >= 4]['NACCID'].iloc[0]
print(f"\nExample patient {example_id} trajectory:")
print(dx[dx['NACCID'] == example_id][
    ['NACCVNUM', 'NACCDAYS', 'NACCUDSD']].sort_values('NACCVNUM'))

Loaded: 214,976 rows, 56,532 patients

NACCDAYS at visit 1 (NACCVNUM==1) — should be ~0:
count    56532.000000
mean      1228.745666
std       1473.705739
min          0.000000
25%          0.000000
50%        737.000000
75%       1871.000000
max       7308.000000
Name: NACCDAYS, dtype: float64

NACCDAYS at visit 2 — should be larger (~365+):
count    39365.000000
mean      1764.599263
std       1474.233121
min          0.000000
25%        694.000000
50%       1270.000000
75%       2428.000000
max       7308.000000
Name: NACCDAYS, dtype: float64

Example patient NACC885822 trajectory:
   NACCVNUM  NACCDAYS  NACCUDSD
1         1      2630         4
2         2      2630         4
3         3      2630         4
4         4      2630         4
5         5      2630         4


In [7]:
# ============================================================
# NACC STEP 2 (CORRECTED) — Use VISITDATE for real visit timing
# ============================================================
# WHY: NACCDAYS turned out to be a per-patient constant (total
# participation span), not per-visit timing. So we use VISITDATE
# — the actual calendar date of each visit — and compute months
# from baseline ourselves, exactly as we did with ADNI's EXAMDATE.

cohort_cols = [
    'NACCID', 'NACCVNUM', 'VISITDATE',
    'NACCUDSD', 'NACCALZD'
]

dx = pd.read_csv(nacc_path, usecols=cohort_cols, low_memory=False)
print(f"Loaded: {len(dx):,} rows, {dx['NACCID'].nunique():,} patients")

# Convert VISITDATE to real dates
dx['VISITDATE'] = pd.to_datetime(dx['VISITDATE'], errors='coerce')
print(f"Rows with unreadable dates: {dx['VISITDATE'].isnull().sum()}")

# Sort each patient's visits chronologically
dx = dx.sort_values(['NACCID', 'VISITDATE']).reset_index(drop=True)

# Verify with an MCI example patient (NACCUDSD==3 at visit 1)
mci_v1 = dx[(dx['NACCVNUM'] == 1) & (dx['NACCUDSD'] == 3)]
example_id = mci_v1['NACCID'].iloc[0]
print(f"\nExample MCI patient {example_id} trajectory:")
example = dx[dx['NACCID'] == example_id][
    ['NACCVNUM', 'VISITDATE', 'NACCUDSD']].copy()
# Compute months from their first visit
baseline_date = example['VISITDATE'].min()
example['months_from_bl'] = (
    (example['VISITDATE'] - baseline_date).dt.days / 30.44).round(1)
print(example.to_string(index=False))

Loaded: 214,976 rows, 56,532 patients
Rows with unreadable dates: 0

Example MCI patient NACC000011 trajectory:
 NACCVNUM  VISITDATE  NACCUDSD  months_from_bl
        1 2006-04-17         3             0.0
        2 2007-06-18         3            14.0
        3 2008-06-03         3            25.6
        4 2009-08-03         2            39.6


In [8]:
# ============================================================
# NACC STEP 3 — Build the MCI cohort and conversion label
# ============================================================
# WHY: Same logic as ADNI, adapted to NACC's coding. For each
# patient who is MCI (NACCUDSD==3) at their first visit, scan
# forward to see if they convert to Dementia (NACCUDSD==4) within
# 24 months. Same parameters as ADNI for a consistent comparison.

WINDOW_MONTHS = 24
GRACE_MONTHS  = 30   # visits up to 30 months still count
MIN_FOLLOWUP  = 18   # need 18+ months to call someone "stable"

# Identify MCI-at-baseline patients: first visit AND NACCUDSD==3
mci_baseline = dx[(dx['NACCVNUM'] == 1) & (dx['NACCUDSD'] == 3)].copy()
print(f"MCI at baseline (visit 1): {len(mci_baseline):,} patients")

# Store each MCI patient's baseline date
mci_patients = mci_baseline[['NACCID', 'VISITDATE']].rename(
    columns={'VISITDATE': 'baseline_date'})

# Build the label for each MCI patient
results = []
for _, patient in mci_patients.iterrows():
    nid = patient['NACCID']
    bl_date = patient['baseline_date']

    # All of this patient's visits, already sorted by date
    visits = dx[dx['NACCID'] == nid].copy()
    visits['months_from_bl'] = (
        (visits['VISITDATE'] - bl_date).dt.days / 30.44)

    # Follow-up visits within the grace window (after baseline)
    window = visits[(visits['months_from_bl'] > 0) &
                    (visits['months_from_bl'] <= GRACE_MONTHS)]

    # Did they convert to Dementia (NACCUDSD==4) in the window?
    converted = (window['NACCUDSD'] == 4).any()
    max_fu = window['months_from_bl'].max() if len(window) > 0 else 0

    if converted:
        conv_month = window[window['NACCUDSD'] == 4]['months_from_bl'].min()
        results.append({'NACCID': nid, 'label': 1,
                        'conversion_month': round(conv_month, 1),
                        'max_followup_months': round(max_fu, 1)})
    elif max_fu >= MIN_FOLLOWUP:
        results.append({'NACCID': nid, 'label': 0,
                        'conversion_month': None,
                        'max_followup_months': round(max_fu, 1)})
    else:
        results.append({'NACCID': nid, 'label': None,
                        'conversion_month': None,
                        'max_followup_months': round(max_fu, 1)})

nacc_cohort = pd.DataFrame(results)

print(f"\nLabel breakdown:")
print(nacc_cohort['label'].value_counts(dropna=False))
labeled = nacc_cohort[nacc_cohort['label'].notna()]
n_conv = (labeled['label'] == 1).sum()
n_stab = (labeled['label'] == 0).sum()
print(f"\nLabeled NACC cohort: {len(labeled):,} patients")
print(f"  Converters: {n_conv:,} ({n_conv/len(labeled)*100:.1f}%)")
print(f"  Stable:     {n_stab:,} ({n_stab/len(labeled)*100:.1f}%)")

MCI at baseline (visit 1): 12,755 patients

Label breakdown:
label
NaN    6434
0.0    4050
1.0    2271
Name: count, dtype: int64

Labeled NACC cohort: 6,321 patients
  Converters: 2,271 (35.9%)
  Stable:     4,050 (64.1%)


In [9]:
# ============================================================
# NACC STEP 4 — Locate the common-feature columns in NACC
# ============================================================
# WHY: We need to map our reduced common-feature model's inputs
# to NACC's actual column names. We search the NACC header for
# each one before extracting, confirming names rather than
# assuming (NACC names differ from ADNI).

preview = pd.read_csv(nacc_path, nrows=5)
all_cols = preview.columns.tolist()

# Search helper: print columns matching any keyword
def find_cols(label, keywords):
    matches = [c for c in all_cols
               if any(k.upper() in c.upper() for k in keywords)]
    print(f"\n{label}:")
    print(f"  {matches}")

find_cols("MMSE (cognitive screen)", ['MMSE'])
find_cols("CDR sum of boxes", ['CDRSUM', 'CDRGLOB'])
find_cols("Functional (FAQ)", ['FAQ', 'FAS'])
find_cols("Depression (GDS)", ['GDS', 'NACCGDS'])
find_cols("Animal fluency", ['ANIMAL'])
find_cols("Logical Memory", ['LOGIMEM', 'MEMUNITS', 'LOGI'])
find_cols("Trails A and B", ['TRAIL'])
find_cols("Education", ['EDUC'])
find_cols("Age", ['NACCAGE'])
find_cols("APOE", ['APOE'])
find_cols("Sex", ['SEX'])
find_cols("Race", ['RACE'])
find_cols("Hispanic ethnicity", ['HISP'])
find_cols("Marital status", ['MARI'])


MMSE (cognitive screen):
  ['MMSECOMP', 'MMSELAN', 'MMSELOC', 'MMSELANX', 'MMSEVIS', 'MMSEHEAR', 'MMSEORDA', 'MMSEORLO', 'NACCMMSE']

CDR sum of boxes:
  ['CDRSUM', 'CDRGLOB']

Functional (FAQ):
  []

Depression (GDS):
  ['NOGDS', 'NACCGDS', 'NGDSGWAS', 'NGDSEXOM', 'NGDSWGS', 'NGDSWES', 'NGDSGWAC', 'NGDSEXAC', 'NGDSWGAC', 'NGDSWEAC']

Animal fluency:
  ['ANIMALS']

Logical Memory:
  ['LOGIMO', 'LOGIDAY', 'LOGIYR', 'LOGIPREV', 'LOGIMEM', 'MEMUNITS']

Trails A and B:
  ['TRAILA', 'TRAILARR', 'TRAILALI', 'TRAILB', 'TRAILBRR', 'TRAILBLI', 'OTRAILA', 'OTRAILB']

Education:
  ['EDUC', 'EXPEDUCINC', 'INEDUC']

Age:
  ['NACCAGE', 'NACCAGEB']

APOE:
  ['NACCAPOE']

Sex:
  ['NACCSEX', 'INTERSEX', 'SEXORNGAY', 'SEXORNHET', 'SEXORNBI', 'SEXORNTWOS', 'SEXORNOTH', 'SEXORNOTHX', 'SEXORNDNK', 'SEXORNNOAN', 'EXPSEXORN', 'INSEX', 'OTHSUBUSEX', 'NGDSEXOM', 'NGDSEXAC', 'NPSEX']

Race:
  ['RACE', 'RACEX', 'RACESEC', 'RACESECX', 'RACETER', 'RACETERX', 'RACEAIAN', 'RACEAIANX', 'RACEASIAN', 'RACEBLACK', 'RAC

In [10]:
# ============================================================
# NACC STEP 5 — Find FAQ and confirm coding of key variables
# ============================================================
# WHY: FAQ wasn't found under obvious names, and several columns
# (APOE, sex, race, marital, GDS) use NACC-specific codes we must
# decode to match ADNI. We locate FAQ and inspect the coding of
# each variable that needs conversion.

preview = pd.read_csv(nacc_path, nrows=5)
all_cols = preview.columns.tolist()

# Broader search for the functional assessment
print("Possible FAQ / functional columns:")
faq_candidates = [c for c in all_cols
                  if any(k in c.upper() for k in
                         ['FAQ', 'FUNC', 'IADL', 'BILLS', 'TAXES',
                          'SHOPPING', 'GAMES', 'STOVE', 'TRAVEL'])]
print(f"  {faq_candidates}")

# Load a sample of the key coded variables to see their values
code_check = pd.read_csv(nacc_path, usecols=[
    'NACCAPOE', 'NACCSEX', 'RACE', 'NACCHISP', 'MARISTAT', 'NACCGDS'
], nrows=5000)

for col in ['NACCAPOE', 'NACCSEX', 'RACE', 'NACCHISP', 'MARISTAT', 'NACCGDS']:
    print(f"\n{col} value distribution (first 5000 rows):")
    print(code_check[col].value_counts(dropna=False).sort_index().head(10))

Possible FAQ / functional columns:
  ['BILLS', 'TAXES', 'SHOPPING', 'GAMES', 'STOVE', 'TRAVEL']

NACCAPOE value distribution (first 5000 rows):
NACCAPOE
1    2418
2    1241
3     374
4     294
5      75
6      22
9     576
Name: count, dtype: int64

NACCSEX value distribution (first 5000 rows):
NACCSEX
1    2164
2    2836
Name: count, dtype: int64

RACE value distribution (first 5000 rows):
RACE
-4      105
 1     3997
 2      680
 3       22
 4        9
 5      102
 50      69
 99      16
Name: count, dtype: int64

NACCHISP value distribution (first 5000 rows):
NACCHISP
0    4543
1     444
9      13
Name: count, dtype: int64

MARISTAT value distribution (first 5000 rows):
MARISTAT
1    2995
2     923
3     615
4      40
5     339
6      75
9      13
Name: count, dtype: int64

NACCGDS value distribution (first 5000 rows):
NACCGDS
-4     471
 0    1496
 1     997
 2     658
 3     406
 4     246
 5     164
 6      91
 7      73
 8      65
Name: count, dtype: int64


In [11]:
# ============================================================
# NACC STEP 6 — Extract & harmonize the common features
# ============================================================
# WHY: Build a NACC feature table matching our model's expected
# inputs exactly. Each variable needs NACC-specific decoding:
# rebuilding FAQ from items, converting APOE genotype codes to
# an allele count, recoding sex/race/ethnicity/marital, and
# converting NACC missing codes (-4, 88, 99, 9) to NaN.

# The 10 FAQ item columns (sum = FAQ total, matching ADNI)
faq_items = ['BILLS', 'TAXES', 'SHOPPING', 'GAMES', 'STOVE',
             'MEALPREP', 'EVENTS', 'PAYATTN', 'REMDATES', 'TRAVEL']

# All NACC columns we need to load
feature_cols = ['NACCID', 'NACCVNUM',
                'NACCMMSE', 'CDRSUM', 'NACCGDS', 'ANIMALS',
                'LOGIMEM', 'MEMUNITS', 'TRAILA', 'TRAILB',
                'EDUC', 'NACCAGE', 'NACCAPOE', 'NACCSEX',
                'RACE', 'NACCHISP', 'MARISTAT'] + faq_items

# Confirm all exist, then load
preview = pd.read_csv(nacc_path, nrows=5)
load_cols = [c for c in feature_cols if c in preview.columns]
missing = [c for c in feature_cols if c not in preview.columns]
if missing:
    print(f"NOTE — not found, will check: {missing}")

feat = pd.read_csv(nacc_path, usecols=load_cols, low_memory=False)

# Take the FIRST visit (baseline) per patient
feat_bl = feat[feat['NACCVNUM'] == 1].drop_duplicates(
    subset='NACCID', keep='first').copy()
print(f"Baseline feature rows: {len(feat_bl):,}")

# --- Convert NACC missing codes to NaN ---
# NACC uses -4, 88, 95-99 as "not available" across many columns.
# We replace these with NaN so they don't corrupt the model.
# We do this per-column with appropriate codes.
def clean_missing(series, codes):
    return series.replace(codes, np.nan)

# Cognitive/continuous: common missing codes are -4, 88, 99, 888, 995-999
cog_missing = [-4, 88, 99, 888, 995, 996, 997, 998, 999]
for col in ['NACCMMSE', 'CDRSUM', 'NACCGDS', 'ANIMALS',
            'LOGIMEM', 'MEMUNITS', 'TRAILA', 'TRAILB', 'EDUC']:
    if col in feat_bl.columns:
        feat_bl[col] = clean_missing(feat_bl[col], cog_missing)

# FAQ items use 0-3 normally, 8=not applicable, 9=unknown -> NaN
for col in faq_items:
    if col in feat_bl.columns:
        feat_bl[col] = clean_missing(feat_bl[col], [8, 9, -4])

# --- Build harmonized features matching ADNI names ---
out = pd.DataFrame()
out['NACCID'] = feat_bl['NACCID']

# Direct numeric maps (just rename)
out['MMSCORE']   = feat_bl['NACCMMSE']
out['CDRSB']     = feat_bl['CDRSUM']
out['GDTOTAL']   = feat_bl['NACCGDS']
out['CATANIMSC'] = feat_bl['ANIMALS']
out['LIMMTOTAL'] = feat_bl['LOGIMEM']
out['LDELTOTAL'] = feat_bl['MEMUNITS']
out['TRAASCOR']  = feat_bl['TRAILA']
out['TRABSCOR']  = feat_bl['TRAILB']
out['PTEDUCAT']  = feat_bl['EDUC']
out['AGE']       = feat_bl['NACCAGE']

# FAQ total: sum the 10 items (skipna=False so partial = missing)
faq_present = [c for c in faq_items if c in feat_bl.columns]
out['FAQTOTAL'] = feat_bl[faq_present].sum(axis=1, skipna=False)

# APOE genotype code -> e4 allele count (0/1/2)
# 1=e3e3(0), 2=e3e4(1), 3=e3e2(0), 4=e4e4(2), 5=e4e2(1), 6=e2e2(0), 9=missing
apoe_to_count = {1: 0, 2: 1, 3: 0, 4: 2, 5: 1, 6: 0, 9: np.nan}
out['APOE4_count'] = feat_bl['NACCAPOE'].map(apoe_to_count)

# Sex: NACCSEX 1=male 2=female -> sex_female
out['sex_female'] = (feat_bl['NACCSEX'] == 2).astype(int)

# Race: RACE 1=White -> race_nonwhite=1 when NOT white; missing codes -> NaN then 0
race_clean = feat_bl['RACE'].replace([-4, 99], np.nan)
out['race_nonwhite'] = (race_clean != 1).astype(int)

# Ethnicity: NACCHISP 1=Hispanic, 9=missing
hisp_clean = feat_bl['NACCHISP'].replace(9, np.nan)
out['ethnicity_hispanic'] = (hisp_clean == 1).astype(int)

# Marital: MARISTAT 1=married, 9=missing
mar_clean = feat_bl['MARISTAT'].replace(9, np.nan)
out['married'] = (mar_clean == 1).astype(int)

print(f"\nHarmonized NACC features: {out.shape[0]} patients, "
      f"{out.shape[1]} columns")
print(f"Columns: {out.columns.tolist()}")

# Coverage check
print(f"\nFeature coverage:")
for col in out.columns:
    if col != 'NACCID':
        have = out[col].notna().sum()
        print(f"  {col:20} {have:5} ({have/len(out)*100:.0f}%)")

Baseline feature rows: 56,532

Harmonized NACC features: 56532 patients, 17 columns
Columns: ['NACCID', 'MMSCORE', 'CDRSB', 'GDTOTAL', 'CATANIMSC', 'LIMMTOTAL', 'LDELTOTAL', 'TRAASCOR', 'TRABSCOR', 'PTEDUCAT', 'AGE', 'FAQTOTAL', 'APOE4_count', 'sex_female', 'race_nonwhite', 'ethnicity_hispanic', 'married']

Feature coverage:
  MMSCORE              32624 (58%)
  CDRSB                56529 (100%)
  GDTOTAL              52672 (93%)
  CATANIMSC            55923 (99%)
  LIMMTOTAL            32624 (58%)
  LDELTOTAL            32624 (58%)
  TRAASCOR             50035 (89%)
  TRABSCOR             44886 (79%)
  PTEDUCAT             56131 (99%)
  AGE                  56532 (100%)
  FAQTOTAL             40382 (71%)
  APOE4_count          44544 (79%)
  sex_female           56532 (100%)
  race_nonwhite        56532 (100%)
  ethnicity_hispanic   56532 (100%)
  married              56532 (100%)


In [12]:
# ============================================================
# NACC STEP 7 — Merge features onto the labeled cohort
# ============================================================
# WHY: The coverage above is across ALL 56,532 NACC patients.
# What matters is coverage within our 6,321 LABELED cohort. We
# merge the harmonized features onto the cohort and check real
# coverage, then see how many patients have complete-enough data
# to be scored by the model.

# Merge harmonized features onto the labeled cohort
labeled = nacc_cohort[nacc_cohort['label'].notna()].copy()
nacc_master = labeled.merge(out, on='NACCID', how='left')

print(f"NACC validation cohort: {len(nacc_master):,} patients")
print(f"  Converters: {(nacc_master['label']==1).sum():,}")
print(f"  Stable:     {(nacc_master['label']==0).sum():,}")

# The features the model expects (the reduced common set)
common_features = [
    'MMSCORE', 'CDRSB', 'FAQTOTAL', 'GDTOTAL', 'CATANIMSC',
    'LIMMTOTAL', 'LDELTOTAL', 'TRAASCOR', 'TRABSCOR',
    'PTEDUCAT', 'AGE', 'APOE4_count',
    'sex_female', 'race_nonwhite', 'ethnicity_hispanic', 'married'
]

print(f"\nFeature coverage WITHIN the labeled cohort:")
for col in common_features:
    have = nacc_master[col].notna().sum()
    print(f"  {col:20} {have:5} ({have/len(nacc_master)*100:.0f}%)")

# How many patients have ALL features present (complete cases)?
complete = nacc_master[common_features].notna().all(axis=1).sum()
print(f"\nPatients with ALL features present: {complete:,} "
      f"({complete/len(nacc_master)*100:.0f}%)")

NACC validation cohort: 6,321 patients
  Converters: 2,271
  Stable:     4,050

Feature coverage WITHIN the labeled cohort:
  MMSCORE               3846 (61%)
  CDRSB                 6321 (100%)
  FAQTOTAL              4033 (64%)
  GDTOTAL               6144 (97%)
  CATANIMSC             6236 (99%)
  LIMMTOTAL             3846 (61%)
  LDELTOTAL             3846 (61%)
  TRAASCOR              5992 (95%)
  TRABSCOR              5698 (90%)
  PTEDUCAT              6304 (100%)
  AGE                   6321 (100%)
  APOE4_count           5593 (88%)
  sex_female            6321 (100%)
  race_nonwhite         6321 (100%)
  ethnicity_hispanic    6321 (100%)
  married               6321 (100%)

Patients with ALL features present: 1,974 (31%)


In [14]:
# ============================================================
# NACC STEP 8 — Check complete-case diversity, then validate
# ============================================================
# WHY: Before trusting the complete-case result, confirm the
# 1,974 patients with full data are still diverse (not a biased,
# ADNI-like subset). Then apply the ADNI-trained reduced model
# and get the primary generalization AUC.

from sklearn.metrics import roc_auc_score, classification_report
import joblib, os

# --- Build the complete-case validation set ---
complete_mask = nacc_master[common_features].notna().all(axis=1)
nacc_complete = nacc_master[complete_mask].copy()

print(f"Complete-case cohort: {len(nacc_complete):,} patients")
print(f"  Converters: {(nacc_complete['label']==1).sum():,} "
      f"({(nacc_complete['label']==1).mean()*100:.1f}%)")

# --- Diversity check: compare to the full labeled cohort ---
print(f"\nDIVERSITY CHECK — complete cases vs full cohort:")
for col, lbl in [('race_nonwhite', 'Non-white'),
                 ('ethnicity_hispanic', 'Hispanic'),
                 ('sex_female', 'Female')]:
    full_pct = nacc_master[col].mean() * 100
    comp_pct = nacc_complete[col].mean() * 100
    print(f"  {lbl:12} full cohort: {full_pct:.1f}%  |  "
          f"complete cases: {comp_pct:.1f}%")
print(f"  {'Mean age':12} full cohort: {nacc_master['AGE'].mean():.1f}  |  "
      f"complete cases: {nacc_complete['AGE'].mean():.1f}")

# Compare to ADNI's 7% non-white for context
print(f"\n  (ADNI cohort was ~7% non-white, ~3.5% Hispanic — "
      f"NACC should be more diverse)")

# --- Apply the ADNI-trained REDUCED model ---
# We must retrain the reduced model on ADNI first (it wasn't saved).
# This requires the ADNI data — so we load the saved reduced model
# if we saved it, OR note we need to build it.
# For now, build predictions assuming the reduced model is available.

# NOTE: we need the ADNI-trained reduced-feature model. If not saved,
# we'll retrain it in the ADNI notebook and save it. Placeholder check:
reduced_model_path = os.path.join(
    r"C:\Users\Chima\OneDrive\Documents\Python Scripts\saved_models",
    'reduced_common_model_LR.joblib')

if os.path.exists(reduced_model_path):
    reduced_model = joblib.load(reduced_model_path)
    X_nacc = nacc_complete[common_features]
    y_nacc = nacc_complete['label'].astype(int)

    proba = reduced_model.predict_proba(X_nacc)[:, 1]
    auc = roc_auc_score(y_nacc, proba)
    pred = reduced_model.predict(X_nacc)

    print(f"\n{'='*55}")
    print(f"PRIMARY NACC VALIDATION (complete cases, n={len(y_nacc)})")
    print(f"{'='*55}")
    print(f"  AUC on NACC: {auc:.3f}")
    print(f"  (ADNI cross-validated AUC was 0.871)")
    print(classification_report(y_nacc, pred,
                                target_names=['Stable', 'Converter']))
else:
    print(f"\n*** The reduced common-feature model isn't saved yet. ***")
    print(f"We need to retrain it on ADNI and save it first.")

Complete-case cohort: 1,974 patients
  Converters: 627 (31.8%)

DIVERSITY CHECK — complete cases vs full cohort:
  Non-white    full cohort: 18.6%  |  complete cases: 17.6%
  Hispanic     full cohort: 7.9%  |  complete cases: 4.8%
  Female       full cohort: 48.2%  |  complete cases: 51.4%
  Mean age     full cohort: 72.6  |  complete cases: 73.2

  (ADNI cohort was ~7% non-white, ~3.5% Hispanic — NACC should be more diverse)

PRIMARY NACC VALIDATION (complete cases, n=1974)
  AUC on NACC: 0.805
  (ADNI cross-validated AUC was 0.871)
              precision    recall  f1-score   support

      Stable       0.83      0.84      0.84      1347
   Converter       0.65      0.63      0.64       627

    accuracy                           0.77      1974
   macro avg       0.74      0.73      0.74      1974
weighted avg       0.77      0.77      0.77      1974



In [15]:
# ============================================================
# NACC STEP 9 — Bootstrap CI for the NACC validation result
# ============================================================
# WHY: Always quantify uncertainty. With n=1,974 the interval
# should be much tighter than ADNI's (which had n=164). We
# bootstrap both AUC and converter recall so we know how much to
# trust each.

from sklearn.metrics import roc_auc_score, recall_score
import numpy as np

X_nacc = nacc_complete[common_features]
y_nacc = nacc_complete['label'].astype(int)
proba = reduced_model.predict_proba(X_nacc)[:, 1]
pred  = reduced_model.predict(X_nacc)

y_arr = y_nacc.values
n = len(y_arr)
rng = np.random.RandomState(42)

auc_boot, recall_boot = [], []
for _ in range(2000):
    idx = rng.randint(0, n, n)
    if len(np.unique(y_arr[idx])) < 2:
        continue
    auc_boot.append(roc_auc_score(y_arr[idx], proba[idx]))
    recall_boot.append(recall_score(y_arr[idx], pred[idx]))

auc_boot, recall_boot = np.array(auc_boot), np.array(recall_boot)

print("="*55)
print(f"NACC VALIDATION — 95% CONFIDENCE INTERVALS (n={n})")
print("="*55)
print(f"\nAUC:    {np.median(auc_boot):.3f}  "
      f"95% CI [{np.percentile(auc_boot,2.5):.3f}, "
      f"{np.percentile(auc_boot,97.5):.3f}]")
print(f"Recall: {np.median(recall_boot):.3f}  "
      f"95% CI [{np.percentile(recall_boot,2.5):.3f}, "
      f"{np.percentile(recall_boot,97.5):.3f}]")
print(f"\nCompare ADNI minimal AUC CI: [0.832, 0.939] (was n=164)")

NACC VALIDATION — 95% CONFIDENCE INTERVALS (n=1974)

AUC:    0.805  95% CI [0.784, 0.826]
Recall: 0.627  95% CI [0.588, 0.666]

Compare ADNI minimal AUC CI: [0.832, 0.939] (was n=164)


In [16]:
# ============================================================
# NACC STEP 10 — Recalibrate the decision threshold on NACC
# ============================================================
# WHY: The 0.5 threshold was tuned to ADNI. We find the threshold
# that best balances sensitivity/specificity on NACC. To avoid
# choosing AND reporting on the same data, we split NACC: pick the
# threshold on a calibration half, report on a held-out half.
# Note: AUC is unchanged by this — only the operating point moves.

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, recall_score, precision_score, roc_auc_score
import numpy as np

X_nacc = nacc_complete[common_features]
y_nacc = nacc_complete['label'].astype(int)

# Split NACC into calibration and reporting halves (stratified)
Xc, Xr, yc, yr = train_test_split(
    X_nacc, y_nacc, test_size=0.5, stratify=y_nacc, random_state=42)

# Probabilities from the ADNI-trained model on each half
proba_c = reduced_model.predict_proba(Xc)[:, 1]
proba_r = reduced_model.predict_proba(Xr)[:, 1]

# --- Find the best threshold on the CALIBRATION half ---
# We use Youden's J statistic: maximizes (sensitivity + specificity - 1),
# a standard, balanced way to choose a threshold.
fpr, tpr, thresholds = roc_curve(yc, proba_c)
youden_j = tpr - fpr
best_threshold = thresholds[np.argmax(youden_j)]
print(f"Best threshold found on calibration half: {best_threshold:.3f}")
print(f"(default was 0.500)")

# --- Apply BOTH thresholds on the held-out REPORTING half ---
pred_default = (proba_r >= 0.5).astype(int)
pred_recal   = (proba_r >= best_threshold).astype(int)

auc_r = roc_auc_score(yr, proba_r)  # unchanged by threshold
print(f"\n{'='*55}")
print(f"REPORTING HALF (n={len(yr)}) — AUC unchanged: {auc_r:.3f}")
print(f"{'='*55}")
print(f"\n{'Metric':12} {'Default (0.5)':>15} {'Recalibrated':>15}")
print(f"{'Recall':12} {recall_score(yr, pred_default):>15.3f} "
      f"{recall_score(yr, pred_recal):>15.3f}")
print(f"{'Precision':12} {precision_score(yr, pred_default):>15.3f} "
      f"{precision_score(yr, pred_recal):>15.3f}")

Best threshold found on calibration half: 0.261
(default was 0.500)

REPORTING HALF (n=987) — AUC unchanged: 0.794

Metric         Default (0.5)    Recalibrated
Recall                 0.610           0.783
Precision              0.639           0.530


In [17]:
# ============================================================
# NACC STEP 11 — Sensitivity check on the full imputed cohort
# ============================================================
# WHY: Our primary result used 1,974 complete cases. This checks
# whether it holds on ALL 6,321 patients, with the pipeline
# imputing missing features. If the AUC is close to 0.805, the
# result is robust and not a complete-case artifact. CI included.

from sklearn.metrics import roc_auc_score
import numpy as np

# Full cohort (pipeline imputes missing values internally)
X_full_nacc = nacc_master[common_features]
y_full_nacc = nacc_master['label'].astype(int)

proba_full = reduced_model.predict_proba(X_full_nacc)[:, 1]
auc_full = roc_auc_score(y_full_nacc, proba_full)

# Bootstrap CI
y_arr = y_full_nacc.values
n = len(y_arr)
rng = np.random.RandomState(42)
boot = []
for _ in range(2000):
    idx = rng.randint(0, n, n)
    if len(np.unique(y_arr[idx])) < 2:
        continue
    boot.append(roc_auc_score(y_arr[idx], proba_full[idx]))
boot = np.array(boot)

print("="*55)
print(f"SENSITIVITY CHECK — full imputed cohort (n={n})")
print("="*55)
print(f"  AUC: {auc_full:.3f}  "
      f"95% CI [{np.percentile(boot,2.5):.3f}, {np.percentile(boot,97.5):.3f}]")
print(f"\n  Primary (complete cases, n=1974): AUC 0.805 [0.784, 0.826]")
print(f"  If these are close, the result is robust.")

SENSITIVITY CHECK — full imputed cohort (n=6321)
  AUC: 0.781  95% CI [0.768, 0.793]

  Primary (complete cases, n=1974): AUC 0.805 [0.784, 0.826]
  If these are close, the result is robust.


In [18]:
# ============================================================
# NACC STEP 12 — Fairness / subgroup analysis
# ============================================================
# WHY: The core motivation for NACC was diversity. We now test
# whether the model performs EQUALLY across demographic subgroups.
# Unequal performance would be a fairness concern; equal
# performance is evidence of equitable generalization. We report
# AUC per subgroup on the complete-case cohort, with sample sizes.

from sklearn.metrics import roc_auc_score
import numpy as np

X_nacc = nacc_complete[common_features]
y_nacc = nacc_complete['label'].astype(int)
proba = reduced_model.predict_proba(X_nacc)[:, 1]

def subgroup_auc(mask, name):
    """AUC within a subgroup, with a quick bootstrap CI."""
    yy = y_nacc[mask].values
    pp = proba[mask]
    if len(np.unique(yy)) < 2 or len(yy) < 30:
        print(f"  {name:24} n={mask.sum():5}  (too few to assess)")
        return
    auc = roc_auc_score(yy, pp)
    rng = np.random.RandomState(42)
    b = []
    for _ in range(1000):
        idx = rng.randint(0, len(yy), len(yy))
        if len(np.unique(yy[idx])) > 1:
            b.append(roc_auc_score(yy[idx], pp[idx]))
    lo, hi = np.percentile(b, [2.5, 97.5])
    print(f"  {name:24} n={mask.sum():5}  AUC={auc:.3f}  [{lo:.3f}, {hi:.3f}]")

print("="*60)
print("FAIRNESS ANALYSIS — AUC by subgroup (complete cases)")
print("="*60)

print("\nBy race:")
subgroup_auc(nacc_complete['race_nonwhite'] == 0, "White")
subgroup_auc(nacc_complete['race_nonwhite'] == 1, "Non-white")

print("\nBy ethnicity:")
subgroup_auc(nacc_complete['ethnicity_hispanic'] == 0, "Not Hispanic")
subgroup_auc(nacc_complete['ethnicity_hispanic'] == 1, "Hispanic")

print("\nBy sex:")
subgroup_auc(nacc_complete['sex_female'] == 0, "Male")
subgroup_auc(nacc_complete['sex_female'] == 1, "Female")

print("\nBy age:")
median_age = nacc_complete['AGE'].median()
subgroup_auc(nacc_complete['AGE'] < median_age, f"Younger (<{median_age:.0f})")
subgroup_auc(nacc_complete['AGE'] >= median_age, f"Older (>={median_age:.0f})")

FAIRNESS ANALYSIS — AUC by subgroup (complete cases)

By race:
  White                    n= 1626  AUC=0.798  [0.775, 0.822]
  Non-white                n=  348  AUC=0.832  [0.781, 0.880]

By ethnicity:
  Not Hispanic             n= 1880  AUC=0.800  [0.780, 0.822]
  Hispanic                 n=   94  AUC=0.911  [0.800, 0.984]

By sex:
  Male                     n=  960  AUC=0.810  [0.777, 0.838]
  Female                   n= 1014  AUC=0.801  [0.774, 0.829]

By age:
  Younger (<73)            n=  894  AUC=0.842  [0.811, 0.873]
  Older (>=73)             n= 1080  AUC=0.771  [0.740, 0.799]


In [21]:
# ============================================================
# NACC STEP 13 — Combined ADNI+NACC training experiment
# ============================================================
# WHY: Test whether adding diverse NACC data to training improves
# performance and narrows the age gap. We hold out a clean NACC
# test set (never seen in training) and evaluate BOTH the
# ADNI-only model and the combined model on it — a fair comparison.

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score
import numpy as np, os

common_features = [
    'MMSCORE', 'CDRSB', 'FAQTOTAL', 'GDTOTAL', 'CATANIMSC',
    'LIMMTOTAL', 'LDELTOTAL', 'TRAASCOR', 'TRABSCOR',
    'PTEDUCAT', 'AGE', 'APOE4_count',
    'sex_female', 'race_nonwhite', 'ethnicity_hispanic', 'married'
]

# --- Load ADNI common-feature data (exported from ADNI notebook) ---
save_folder = r"C:\Users\Chima\OneDrive\Documents\Python Scripts\saved_models"
adni = pd.read_csv(os.path.join(save_folder, 'adni_common_features.csv'))
X_adni = adni[common_features]
y_adni = adni['label'].astype(int)
print(f"ADNI loaded: {len(X_adni)} patients")

# --- NACC complete cases ---
X_nacc_all = nacc_complete[common_features]
y_nacc_all = nacc_complete['label'].astype(int)

# --- Split NACC: training portion + clean held-out test ---
# The held-out test is NEVER used in training — the honest yardstick
X_nacc_tr, X_nacc_te, y_nacc_tr, y_nacc_te = train_test_split(
    X_nacc_all, y_nacc_all, test_size=0.3, stratify=y_nacc_all,
    random_state=42)
print(f"NACC split: {len(X_nacc_tr)} train, {len(X_nacc_te)} held-out test")

# --- Build the two models ---
def make_model():
    return Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced',
                                     max_iter=1000, random_state=42))])

# Model 1: ADNI-only (trained on ADNI alone)
model_adni = make_model()
model_adni.fit(X_adni, y_adni)

# Model 2: Combined (ADNI + NACC-train portion)
X_combined = pd.concat([X_adni, X_nacc_tr], ignore_index=True)
y_combined = pd.concat([y_adni, y_nacc_tr], ignore_index=True)
model_combined = make_model()
model_combined.fit(X_combined, y_combined)
print(f"Combined model trained on {len(X_combined)} patients "
      f"(ADNI {len(X_adni)} + NACC {len(X_nacc_tr)})")

# --- Evaluate BOTH on the same held-out NACC test ---
def eval_with_ci(model, X, y, label):
    proba = model.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, proba)
    yv = y.values; n = len(yv); rng = np.random.RandomState(42)
    b = [roc_auc_score(yv[i], proba[i]) for i in
         (rng.randint(0, n, n) for _ in range(2000))
         if len(np.unique(yv[i])) > 1]
    lo, hi = np.percentile(b, [2.5, 97.5])
    print(f"  {label:18} AUC={auc:.3f}  95% CI [{lo:.3f}, {hi:.3f}]")
    return proba

print(f"\n{'='*55}")
print(f"FAIR COMPARISON on held-out NACC test (n={len(y_nacc_te)})")
print(f"{'='*55}")
proba_adni = eval_with_ci(model_adni, X_nacc_te, y_nacc_te, "ADNI-only")
proba_comb = eval_with_ci(model_combined, X_nacc_te, y_nacc_te, "Combined")

# --- Re-check the age gap with the combined model ---
print(f"\n{'='*55}")
print(f"AGE GAP on held-out NACC test")
print(f"{'='*55}")
med_age = X_nacc_te['AGE'].median()
for grp_name, grp_mask in [("Younger", X_nacc_te['AGE'] < med_age),
                            ("Older",   X_nacc_te['AGE'] >= med_age)]:
    yy = y_nacc_te[grp_mask].values
    if len(np.unique(yy)) > 1:
        a_adni = roc_auc_score(yy, proba_adni[grp_mask.values])
        a_comb = roc_auc_score(yy, proba_comb[grp_mask.values])
        print(f"  {grp_name:8} (n={grp_mask.sum():4})  "
              f"ADNI-only={a_adni:.3f}  Combined={a_comb:.3f}")

ADNI loaded: 816 patients
NACC split: 1381 train, 593 held-out test
Combined model trained on 2197 patients (ADNI 816 + NACC 1381)

FAIR COMPARISON on held-out NACC test (n=593)
  ADNI-only          AUC=0.791  95% CI [0.752, 0.830]
  Combined           AUC=0.806  95% CI [0.769, 0.841]

AGE GAP on held-out NACC test
  Younger  (n= 283)  ADNI-only=0.846  Combined=0.855
  Older    (n= 310)  ADNI-only=0.741  Combined=0.754


In [22]:
# ============================================================
# Real-world single-patient prediction tool
# ============================================================
# WHY: Simulates a doctor entering one new MCI patient's baseline
# data and getting a conversion-risk estimate. This is the core
# logic that would power an interactive clinical app. The model
# takes the 16 common features and returns the probability of
# converting to dementia within 24 months.

import pandas as pd
import numpy as np

def predict_patient(patient_dict, model, feature_list):
    """
    Predict conversion risk for a single new MCI patient.
    patient_dict : dictionary of {feature_name: value}
    model        : a trained pipeline (handles imputation/scaling)
    feature_list : the features the model expects, in order
    """
    # Build a one-row dataframe in the exact feature order the
    # model expects. Any feature the doctor didn't provide is set
    # to NaN — the pipeline's imputer fills it from training medians.
    row = {f: patient_dict.get(f, np.nan) for f in feature_list}
    X_new = pd.DataFrame([row])[feature_list]

    # Get the conversion probability
    prob_convert = model.predict_proba(X_new)[0, 1]

    # Report which features the doctor actually provided vs imputed
    provided = [f for f in feature_list if f in patient_dict]
    imputed  = [f for f in feature_list if f not in patient_dict]

    print("="*55)
    print("SINGLE PATIENT CONVERSION RISK ESTIMATE")
    print("="*55)
    print(f"\n  Probability of converting to dementia within 24 months:")
    print(f"      {prob_convert*100:.1f}%")
    print(f"  Probability of remaining stable:")
    print(f"      {(1-prob_convert)*100:.1f}%")

    # A plain-language risk band for interpretability
    if prob_convert >= 0.66:
        band = "HIGH risk — consider close monitoring / early intervention"
    elif prob_convert >= 0.33:
        band = "MODERATE risk — warrants follow-up"
    else:
        band = "LOWER risk — routine monitoring"
    print(f"\n  Interpretation: {band}")

    if imputed:
        print(f"\n  NOTE: {len(imputed)} feature(s) not provided were "
              f"estimated from training data:")
        print(f"        {imputed}")
        print(f"  Fewer missing inputs = more reliable estimate.")
    print()
    return prob_convert

# --- EXAMPLE: a hypothetical new MCI patient ---
# A doctor enters what they measured. Realistic values for a
# higher-risk MCI patient (poor memory, smaller cognition scores).
example_patient = {
    'MMSCORE': 26,          # MMSE (lower = worse)
    'CDRSB': 2.5,           # CDR sum of boxes
    'FAQTOTAL': 8,          # functional impairment
    'GDTOTAL': 3,           # depression
    'CATANIMSC': 14,        # animal fluency
    'LIMMTOTAL': 8,         # logical memory immediate
    'LDELTOTAL': 4,         # logical memory delayed (low = concerning)
    'TRAASCOR': 45,         # trails A (seconds)
    'TRABSCOR': 140,        # trails B (seconds)
    'PTEDUCAT': 16,         # years education
    'AGE': 74,
    'APOE4_count': 1,       # one e4 allele
    'sex_female': 1,
    'race_nonwhite': 0,
    'ethnicity_hispanic': 0,
    'married': 1,
}

predict_patient(example_patient, reduced_model, common_features)

SINGLE PATIENT CONVERSION RISK ESTIMATE

  Probability of converting to dementia within 24 months:
      88.1%
  Probability of remaining stable:
      11.9%

  Interpretation: HIGH risk — consider close monitoring / early intervention



np.float64(0.8812307954078927)

In [23]:
# ============================================================
# Dual-model single-patient prediction tool
# ============================================================
# WHY: Embodies the project's thesis — two models for two
# resource levels. If the patient has the full workup (RAVLT,
# ADAS, MRI), use the richer 26-feature model. If only basic
# tests are available, use the accessible 16-feature model. The
# tool auto-selects based on which features are provided.

import pandas as pd
import numpy as np
import joblib, os

save_folder = r"C:\Users\Chima\OneDrive\Documents\Python Scripts\saved_models"

# Load both models and their feature lists
full_model     = joblib.load(os.path.join(save_folder, 'minimal_model_LR.joblib'))
full_features  = joblib.load(os.path.join(save_folder, 'minimal_features.joblib'))
reduced_model_loaded = joblib.load(os.path.join(save_folder, 'reduced_common_model_LR.joblib'))
reduced_features = [
    'MMSCORE', 'CDRSB', 'FAQTOTAL', 'GDTOTAL', 'CATANIMSC',
    'LIMMTOTAL', 'LDELTOTAL', 'TRAASCOR', 'TRABSCOR',
    'PTEDUCAT', 'AGE', 'APOE4_count',
    'sex_female', 'race_nonwhite', 'ethnicity_hispanic', 'married'
]

# The features that exist ONLY in the full (26-feature) model —
# the "premium" inputs that require a full workup.
full_only_features = [f for f in full_features if f not in reduced_features]
print(f"Full-only (premium) features: {full_only_features}\n")


def predict_patient_auto(patient_dict, verbose=True):
    """
    Auto-selects the model based on available data.
    If the patient has enough of the premium features (RAVLT, ADAS,
    MRI), use the richer 26-feature model. Otherwise fall back to
    the accessible 16-feature model.
    """
    # Which premium features did the doctor provide?
    premium_provided = [f for f in full_only_features
                        if f in patient_dict and patient_dict[f] is not None]

    # Decision rule: if the patient has MOST premium features
    # (here: at least half), use the full model; else the reduced.
    use_full = len(premium_provided) >= (len(full_only_features) / 2)

    if use_full:
        model, feats, name, auc = (full_model, full_features,
                                   "26-FEATURE (full workup)", "0.887")
    else:
        model, feats, name, auc = (reduced_model_loaded, reduced_features,
                                   "16-FEATURE (accessible)", "0.871 / 0.805 ext.")

    # Build the input row in the model's expected feature order
    row = {f: patient_dict.get(f, np.nan) for f in feats}
    X_new = pd.DataFrame([row])[feats]
    prob = model.predict_proba(X_new)[0, 1]

    # Which provided features were actually used, which imputed
    used    = [f for f in feats if f in patient_dict
               and patient_dict[f] is not None]
    imputed = [f for f in feats if f not in patient_dict
               or patient_dict.get(f) is None]

    if verbose:
        print("="*55)
        print("CONVERSION RISK ESTIMATE")
        print("="*55)
        print(f"  Model selected: {name}  (AUC {auc})")
        print(f"  Reason: {len(premium_provided)}/{len(full_only_features)} "
              f"premium features provided")
        print(f"\n  Probability of converting to dementia "
              f"within 24 months: {prob*100:.1f}%")
        print(f"  Probability of remaining stable: {(1-prob)*100:.1f}%")

        if prob >= 0.66:
            band = "HIGH — closer monitoring / intervention planning"
        elif prob >= 0.33:
            band = "MODERATE — attentive follow-up"
        else:
            band = "LOWER — routine monitoring (within 24-month window)"
        print(f"\n  Interpretation: {band}")

        if imputed:
            print(f"\n  NOTE: {len(imputed)} feature(s) estimated from "
                  f"training data (fewer = more reliable):")
            print(f"        {imputed}")
        print()
    return prob, name


# --- EXAMPLE 1: Patient with FULL workup (routes to 26-feature) ---
print(">>> EXAMPLE 1: Patient with complete workup\n")
patient_full = {
    'TOTSCORE': 18, 'TOTAL13': 28,           # ADAS
    'MMSCORE': 26, 'CDRSB': 2.5, 'FAQTOTAL': 8, 'GDTOTAL': 3,
    'RAVLT_learning_total': 28, 'AVDEL30MIN': 3, 'AVDELTOT': 4,  # RAVLT
    'TRAASCOR': 45, 'TRABSCOR': 140, 'CATANIMSC': 14,
    'LIMMTOTAL': 8, 'LDELTOTAL': 4,
    'Hippocampus_total': 6200, 'Entorhinal_total': 3100,         # MRI
    'Ventricle_total': 42000, 'Fusiform_total': 16000, 'ICV': 1450000,
    'PTEDUCAT': 16, 'AGE': 74, 'APOE4_count': 1,
    'sex_female': 1, 'race_nonwhite': 0, 'ethnicity_hispanic': 0, 'married': 1,
}
predict_patient_auto(patient_full)

# --- EXAMPLE 2: Patient with only BASIC tests (routes to 16-feature) ---
print(">>> EXAMPLE 2: Patient with only basic/accessible tests\n")
patient_basic = {
    'MMSCORE': 26, 'CDRSB': 2.5, 'FAQTOTAL': 8, 'GDTOTAL': 3,
    'CATANIMSC': 14, 'LIMMTOTAL': 8, 'LDELTOTAL': 4,
    'TRAASCOR': 45, 'TRABSCOR': 140,
    'PTEDUCAT': 16, 'AGE': 74, 'APOE4_count': 1,
    'sex_female': 1, 'race_nonwhite': 0, 'ethnicity_hispanic': 0, 'married': 1,
}
predict_patient_auto(patient_basic)

Full-only (premium) features: ['TOTSCORE', 'TOTAL13', 'RAVLT_learning_total', 'AVDEL30MIN', 'AVDELTOT', 'Hippocampus_total', 'Entorhinal_total', 'Ventricle_total', 'Fusiform_total', 'ICV']

>>> EXAMPLE 1: Patient with complete workup

CONVERSION RISK ESTIMATE
  Model selected: 26-FEATURE (full workup)  (AUC 0.887)
  Reason: 10/10 premium features provided

  Probability of converting to dementia within 24 months: 91.9%
  Probability of remaining stable: 8.1%

  Interpretation: HIGH — closer monitoring / intervention planning

>>> EXAMPLE 2: Patient with only basic/accessible tests

CONVERSION RISK ESTIMATE
  Model selected: 16-FEATURE (accessible)  (AUC 0.871 / 0.805 ext.)
  Reason: 0/10 premium features provided

  Probability of converting to dementia within 24 months: 88.1%
  Probability of remaining stable: 11.9%

  Interpretation: HIGH — closer monitoring / intervention planning



(np.float64(0.8812307954078927), '16-FEATURE (accessible)')

In [24]:
# ============================================================
# Package the complete dual-model system into one bundle
# ============================================================
# WHY: Bundle both models, their feature lists, routing logic,
# performance metadata, and the recalibrated threshold into a
# single self-describing object. This makes the system fully
# reproducible and ready to power an app — one file, everything
# needed to make and interpret predictions.

import joblib, os
from datetime import datetime

save_folder = r"C:\Users\Chima\OneDrive\Documents\Python Scripts\saved_models"

# Load the pieces (in case they're not all in memory)
full_model       = joblib.load(os.path.join(save_folder, 'minimal_model_LR.joblib'))
full_features    = joblib.load(os.path.join(save_folder, 'minimal_features.joblib'))
reduced_model    = joblib.load(os.path.join(save_folder, 'reduced_common_model_LR.joblib'))

reduced_features = [
    'MMSCORE', 'CDRSB', 'FAQTOTAL', 'GDTOTAL', 'CATANIMSC',
    'LIMMTOTAL', 'LDELTOTAL', 'TRAASCOR', 'TRABSCOR',
    'PTEDUCAT', 'AGE', 'APOE4_count',
    'sex_female', 'race_nonwhite', 'ethnicity_hispanic', 'married'
]
full_only_features = [f for f in full_features if f not in reduced_features]

# Build the bundle as a single dictionary
mci_predictor = {
    # The two trained models
    'full_model': full_model,            # 26-feature, full workup
    'reduced_model': reduced_model,      # 16-feature, accessible

    # Feature lists each model expects (in order)
    'full_features': full_features,
    'reduced_features': reduced_features,
    'full_only_features': full_only_features,  # premium routing features

    # How to use it: routing rule + recalibrated threshold
    'routing_rule': 'Use full_model if >=50% of full_only_features present, else reduced_model',
    'default_threshold': 0.5,
    'nacc_recalibrated_threshold': 0.261,  # for accessible model on diverse pops

    # Performance metadata (so the numbers travel with the models)
    'performance': {
        'full_model_adni_cv_auc': 0.887,
        'reduced_model_adni_cv_auc': 0.871,
        'reduced_model_nacc_auc': 0.805,
        'reduced_model_nacc_auc_ci': (0.784, 0.826),
        'reduced_model_nacc_recall_recalibrated': 0.783,
    },

    # Documentation
    'task': 'Predict conversion from MCI to dementia within 24 months',
    'output': 'Probability of conversion (0-1); complement = probability stable',
    'note': 'Prognostic tool for already-diagnosed MCI patients; not a diagnostic classifier',
    'created': datetime.now().strftime('%Y-%m-%d'),
}

# Save the complete bundle
bundle_path = os.path.join(save_folder, 'mci_conversion_predictor.joblib')
joblib.dump(mci_predictor, bundle_path)

# Confirm and report
size_kb = os.path.getsize(bundle_path) / 1024
print(f"Packaged system saved: mci_conversion_predictor.joblib ({size_kb:.0f} KB)")
print(f"\nBundle contents:")
for key in mci_predictor:
    val = mci_predictor[key]
    if key in ('full_model', 'reduced_model'):
        print(f"  {key}: <trained pipeline>")
    elif isinstance(val, dict):
        print(f"  {key}: {{...{len(val)} items}}")
    elif isinstance(val, list):
        print(f"  {key}: [{len(val)} items]")
    else:
        print(f"  {key}: {val}")

Packaged system saved: mci_conversion_predictor.joblib (6 KB)

Bundle contents:
  full_model: <trained pipeline>
  reduced_model: <trained pipeline>
  full_features: [26 items]
  reduced_features: [16 items]
  full_only_features: [10 items]
  routing_rule: Use full_model if >=50% of full_only_features present, else reduced_model
  default_threshold: 0.5
  nacc_recalibrated_threshold: 0.261
  performance: {...5 items}
  task: Predict conversion from MCI to dementia within 24 months
  output: Probability of conversion (0-1); complement = probability stable
  note: Prognostic tool for already-diagnosed MCI patients; not a diagnostic classifier
  created: 2026-06-25


In [25]:
# How to reload and use the packaged system later
import joblib
predictor = joblib.load('mci_conversion_predictor.joblib')

# Access any component
model = predictor['full_model']
features = predictor['full_features']
# ...then predict as before

FileNotFoundError: [Errno 2] No such file or directory: 'mci_conversion_predictor.joblib'

In [26]:
# How to reload and use the packaged system later
import joblib, os

save_folder = r"C:\Users\Chima\OneDrive\Documents\Python Scripts\saved_models"
bundle_path = os.path.join(save_folder, 'mci_conversion_predictor.joblib')

predictor = joblib.load(bundle_path)

# Confirm it loaded and peek at the contents
print("Bundle loaded successfully.")
print(f"Models inside: full_model, reduced_model")
print(f"Task: {predictor['task']}")
print(f"Full model AUC: {predictor['performance']['full_model_adni_cv_auc']}")

# Access any component
model = predictor['full_model']
features = predictor['full_features']

Bundle loaded successfully.
Models inside: full_model, reduced_model
Task: Predict conversion from MCI to dementia within 24 months
Full model AUC: 0.887


In [27]:
# ============================================================
# Save the already-trained combined model from memory
# ============================================================
# WHY: model_combined is still in memory from the experiment.
# We save that exact fitted object directly — no retraining.

import joblib, os
save_folder = r"C:\Users\Chima\OneDrive\Documents\Python Scripts\saved_models"

joblib.dump(model_combined,
            os.path.join(save_folder, 'combined_adni_nacc_model_LR.joblib'))
print("Combined ADNI+NACC model saved directly from memory.")

# Confirm all your saved models are now present
print("\nAll saved models:")
for f in sorted(os.listdir(save_folder)):
    if f.endswith('.joblib'):
        print(f"  {f}")

Combined ADNI+NACC model saved directly from memory.

All saved models:
  combined_adni_nacc_model_LR.joblib
  full_features.joblib
  full_model_RF.joblib
  mci_conversion_predictor.joblib
  minimal_features.joblib
  minimal_model_LR.joblib
  reduced_common_model_LR.joblib
